In [ ]:
##################################################################
# # ! Icosphere
##################################################################
# %%
# # ! Setup
import pyvista as pv;
import numpy as np;
import pandas as pd;
import time;
from gravity_forward_numpy import *;
from gravity_forward_numba import *;

In [ ]:
# # ! Icosphere <Geometry>
xc, yc, zc = 0.1, 0.0, 2.0;
a = 1.5;
vol = 4*np.pi*a**3/3; area = 4*np.pi*a**2;
nsub_max = 10;
Nsubs = np.arange(nsub_max);
NVs = np.zeros(nsub_max, dtype=int);
NFs = np.zeros(nsub_max, dtype=int);
Es_vol = np.zeros(nsub_max);
Es_area = np.zeros(nsub_max);
MINs_psi = np.zeros(nsub_max);
MAXs_psi = np.zeros(nsub_max);
for insub, nsub in enumerate(Nsubs):
    icosph = pv.Icosphere(radius=a, center=(xc, yc, zc), nsub=nsub);
    Verts  = icosph.points;
    Faces = icosph.regular_faces;
    NVs[insub] = Verts.shape[0];
    NFs[insub] = Faces.shape[0];
    Es_vol[insub] = 1. - icosph.volume/vol;
    Es_area[insub] = 1. - icosph.area/area;
    vert_unit = (Verts - np.array([xc, yc, zc]))/a;
    min_psi, max_psi = spherical_edge_length_range(vert_unit, Faces);
    MINs_psi[insub] = 60. * np.rad2deg(min_psi); # arc-min
    MAXs_psi[insub] = 60. * np.rad2deg(max_psi);
print(f"{'nsub':>4} {'N_vertices':>10} {'N_faces':>10} "
      f"{'Vol_err':>10} {'Area_err':>10} {'Min_psi':>10} {'Max_psi':>10}")
print("-" * 63)
for i in range(len(Nsubs)):
    print(f"{Nsubs[i]:>4} {int(NVs[i]):>10} {int(NFs[i]):>10} "
          f"{Es_vol[i]:>10.2e} {Es_area[i]:>10.2e} "
          f"{MINs_psi[i]:>10.2f} {MAXs_psi[i]:>10.2f}")

nsub N_vertices    N_faces    Vol_err   Area_err    Min_psi    Max_psi
---------------------------------------------------------------
   0         12         20   3.95e-01   2.38e-01    3806.10    3806.10
   1         42         80   1.27e-01   7.17e-02    1903.05    2160.00
   2        162        320   3.38e-02   1.89e-02     872.72    1121.97
   3        642       1280   8.62e-03   4.79e-03     410.91     566.66
   4       2562       5120   2.17e-03   1.20e-03     198.82     284.05
   5      10242      20480   5.42e-04   3.01e-04      97.74     142.12
   6      40962      81920   1.36e-04   7.52e-05      48.44      71.08
   7     163842     327680   3.39e-05   1.88e-05      24.08      35.61
   8     655362    1310720   8.47e-06   4.70e-06      11.91      17.92
   9    2621442    5242880   2.12e-06   1.18e-06       5.74       9.21


In [ ]:
# # ! Icosphere <Gravity>
rho = 1000.;
xgv = np.linspace(-10., 10., 51);
ygv = np.linspace(-10., 10., 51);
z0 = 0.;
[X2d, Y2d] = np.meshgrid(xgv, ygv);
Z2d = z0 * np.ones(X2d.shape);
XPs, YPs, ZPs = map(lambda x: x.flatten(), [X2d, Y2d, Z2d]);
P = np.column_stack((XPs, YPs, ZPs));
######## * Sphere
t1 = time.time();
V, gx, gy, gz, Txx, Tyy, Tzz, Txy, Txz, Tyz \
    = gsphere(P[:,0], P[:,1], P[:,2], xc, yc, zc, a, rho);
tc_sphere = time.time() - t1;
print(f'<Sphere> time cost: {tc_sphere:.2f} sec');
######## * Polyhedron
t1 = time.time();
V_cal, gx_cal, gy_cal, gz_cal, \
Txx_cal, Tyy_cal, Tzz_cal, Txy_cal, Txz_cal, Tyz_cal \
    = VecWerSch_numba(P, Verts, Faces, rho);
tc_numba = time.time() - t1;
print(f'<Polyhedron> time cost: {tc_numba:.2f} sec');

<Sphere> time cost: 0.00 sec
<Polyhedron> time cost: 105.27 sec


In [ ]:
# ! #  Pandas disp stats 
fields = ['V', 'gx', 'gy', 'gz', 'Txx', 'Tyy', 'Tzz', 'Txy', 'Txz', 'Tyz']

df_ref = pd.DataFrame({
    name: {'Min': arr.min(), 'Max': arr.max(), 'Mean': arr.mean(), 'Std': arr.std()}
    for name, arr in zip(fields, [V, gx, gy, gz, Txx, Tyy, Tzz, Txy, Txz, Tyz])
}).T
df_cal = pd.DataFrame({
    name: {'Min': arr.min(), 'Max': arr.max(), 'Mean': arr.mean(), 'Std': arr.std()}
    for name, arr in zip(fields, [V_cal, gx_cal, gy_cal, gz_cal, 
                                  Txx_cal, Tyy_cal, Tzz_cal, Txy_cal, Txz_cal, Tyz_cal])
}).T
df_diff = df_cal - df_ref

def print_table(df, title):
    print(f"\n{title}")
    print("=" * len(title))
    print(df.to_string(float_format="{:12.6f}".format))
print_table(df_ref, "Reference")
print_table(df_cal, "Polyhedron")
print_table(df_diff, "Difference")


Reference
             Min          Max         Mean          Std
V       0.065739     0.471190     0.137072     0.069274
gx     -9.058147     9.037391     0.005996     1.987109
gy     -8.964788     8.964788     0.000000     1.987129
gz      0.063821    23.500740     1.177052     2.647185
Txx  -116.624622    23.843904    -0.599702    10.838320
Tyy  -117.503702    23.750439    -0.599600    10.838333
Tzz    -4.219713   234.128325     1.199301    17.703851
Txy   -32.764925    32.764925    -0.000000     6.262086
Txz  -100.511959   100.415136     0.002784    12.549957
Tyz   -97.973664    97.973664    -0.000000    12.549958

Polyhedron
             Min          Max         Mean          Std
V       0.065739     0.471189     0.137072     0.069274
gx     -9.058127     9.037371     0.005996     1.987105
gy     -8.964769     8.964769     0.000000     1.987125
gz      0.063821    23.500690     1.177050     2.647179
Txx  -116.624383    23.843854    -0.599700    10.838297
Tyy  -117.503433    23.75

In [ ]:
df_ref

,Min,Max,Mean,Std
V,0.065739,0.471190,1.370723e-01,0.069274
gx,-9.058147,9.037391,5.996482e-03,1.987109
gy,-8.964788,8.964788,0.000000e+00,1.987129
gz,0.063821,23.500740,1.177052e+00,2.647185
Txx,-116.624622,23.843904,-5.997016e-01,10.838320
Tyy,-117.503702,23.750439,-5.995999e-01,10.838333
Tzz,-4.219713,234.128325,1.199301e+00,17.703851
Txy,-32.764925,32.764925,-6.146563e-17,6.262086
Txz,-100.511959,100.415136,2.784323e-03,12.549957
Tyz,-97.973664,97.973664,-3.496712e-16,12.549958


In [ ]:
df_cal

,Min,Max,Mean,Std
V,0.065739,0.471189,1.370720e-01,0.069274
gx,-9.058127,9.037371,5.996469e-03,1.987105
gy,-8.964769,8.964769,6.643752e-15,1.987125
gz,0.063821,23.500690,1.177050e+00,2.647179
Txx,-116.624383,23.843854,-5.997003e-01,10.838297
Tyy,-117.503433,23.750389,-5.995986e-01,10.838310
Tzz,-4.219704,234.127816,1.199299e+00,17.703814
Txy,-32.764856,32.764856,3.727549e-15,6.262073
Txz,-100.511738,100.414913,2.784318e-03,12.549930
Tyz,-97.973465,97.973465,-1.923191e-14,12.549932


In [ ]:
df_diff

,Min,Max,Mean,Std
V,-1.393616e-07,-0.000001,-2.905823e-07,-1.468573e-07
gx,1.930181e-05,-0.000019,-1.271161e-08,-4.212258e-06
gy,1.886290e-05,-0.000019,6.643752e-15,-4.214273e-06
gz,-1.352958e-07,-0.000050,-2.495255e-06,-5.612920e-06
Txx,2.395820e-04,-0.000051,1.271318e-06,-2.292852e-05
Tyy,2.691668e-04,-0.000050,1.271106e-06,-2.309180e-05
Tzz,8.945063e-06,-0.000509,-2.542423e-06,-3.759543e-05
Txy,6.954284e-05,-0.000070,3.789015e-15,-1.330806e-05
Txz,2.210668e-04,-0.000223,-5.830753e-09,-2.658002e-05
Tyz,1.994357e-04,-0.000199,-1.888224e-14,-2.672098e-05
